# 🏥 Insurance AI Lab — Prompt Engineering & OpenAI API

> **Workshop:** Prompt Engineering Series  
> **Use Case:** InsureCo Claims Portal  
> **Platform:** AWS EC2 Ubuntu · Jupyter Notebook  
> **Prerequisite:** OpenAI API Key

---

## 📋 Modules in This Notebook

| Module | Topic | Key Skills |
|--------|-------|------------|
| 1 | Claims Classification | Zero-shot · Few-shot · Role + CoT |
| 2 | Structured JSON Output | Function Calling · JSON Schema |
| 3 | Dataset Insights & KPIs | Pandas + GPT · SQL Gen · Forecasting |
| 4 | Prompt Refinement | Constraints · Self-critique · Retry · Temperature |

---

### Business Context
InsureCo processes thousands of insurance claims daily. This lab builds an end-to-end AI pipeline that:
- Classifies claims as **genuine** or **fraud**
- Extracts **structured JSON fields** for downstream systems
- Generates **business KPIs, insights, SQL queries, and forecasts** from data
- Applies **systematic prompt refinement** to fix incorrect outputs


---
# 🟢 Module 1 — Claims Classification

**Goal:** Build a progressive classification pipeline that labels insurance claim text as `genuine` or `fraud`.

We move through three prompting strategies and measure accuracy at each step:

```
Zero-Shot (baseline) → Few-Shot → Role + Chain-of-Thought
```


## 📦 Step 1 — Install Dependencies & Initialise OpenAI Client


In [ ]:
# Install required packages (run once)
# !pip install openai pandas numpy matplotlib seaborn -q

import os, json, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from openai import OpenAI

warnings.filterwarnings('ignore')

# ─── Set your OpenAI API Key ───────────────────────────────────
# Option A: paste directly (development only)
OPENAI_API_KEY = 'sk-...'   # ← replace with your key

# Option B: read from environment (recommended for production)
# OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY')

client = OpenAI(api_key=OPENAI_API_KEY)

# Quick connectivity test
resp = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[{'role': 'user', 'content': 'Reply with one word: Ready'}],
    max_tokens=5
)
print('✅ OpenAI API connected:', resp.choices[0].message.content.strip())


## 🗃️ Step 2 — Generate the Synthetic Claims Dataset


In [ ]:
import random, datetime

random.seed(42)
np.random.seed(42)

claim_texts = [
    # ── Genuine claims ──────────────────────────────────────────────────
    'My car was rear-ended at a red light on 12th Ave. The other driver admitted fault and a police report was filed.',
    'A burst pipe flooded the kitchen overnight. The plumber confirmed extensive water damage to the flooring and cabinets.',
    'I slipped on icy pavement outside the grocery store and fractured my wrist. Hospital records and X-rays are attached.',
    'A hailstorm caused multiple dents across the hood and roof of my 2021 Toyota Camry. Photos are attached.',
    'A fire started in the garage due to faulty wiring. Two rooms and the garage were damaged. Fire department report enclosed.',
    'My bicycle was stolen from a locked rack outside the office building. I filed a police report the same evening.',
    'A large tree fell on the fence and part of the roof during last Tuesday night storm. Contractor estimate attached.',
    'Minor fender-bender in a shopping centre parking lot. Both drivers exchanged insurance details and took photos.',
    # ── Fraudulent claims ────────────────────────────────────────────────
    'I lost my wedding ring at the beach last weekend. I only realized how much it was worth after checking current prices.',
    'My car was broken into while parked downtown, but I forgot to mention I had left the windows slightly open.',
    'My laptop was stolen from my home office. I cannot find the receipt but I remember it was very expensive when I bought it.',
    'There was a fire in the living room but the fire department found absolutely no electrical faults or ignition source.',
    'I was injured from a slip at work, but three coworkers have stated I actually tripped over my own laptop bag.',
    'My phone was damaged by water. It actually stopped working two weeks before the alleged incident date.',
    'My vehicle was hit while parked on a busy street, but there are no witnesses and no CCTV in a normally busy area.',
    'This is my second claim for loss of the same watch within six months. No receipt for either claim.',
]

labels = ['genuine'] * 8 + ['fraud'] * 8

amounts = [random.randint(800, 25000) for _ in range(16)]
dates   = [datetime.date(2024, random.randint(1, 12), random.randint(1, 28)) for _ in range(16)]
policy_ids = [f'POL-{7000 + i}' for i in range(16)]

df = pd.DataFrame({
    'claim_id':   [f'CLM-{1000 + i}' for i in range(16)],
    'policy_id':  policy_ids,
    'date':       dates,
    'claim_text': claim_texts,
    'amount_usd': amounts,
    'true_label': labels,
})

df.to_csv('claims_dataset.csv', index=False)
print(f'✅ Dataset created — {len(df)} claims ({df["true_label"].value_counts().to_dict()})')
df[['claim_id', 'date', 'amount_usd', 'true_label']].head(10)


## 🔍 Step 3 — Zero-Shot Classification (Baseline)

**Technique:** Provide only a task description — no examples, no persona.

This is our **accuracy baseline**. The model uses general knowledge to classify.


In [ ]:
def classify_zero_shot(text):
    prompt = f"""You are an insurance claims analyst.
Classify the following claim as either 'genuine' or 'fraud'.
Respond with exactly one word: genuine  or  fraud.

Claim: {text}
Classification:"""

    resp = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': prompt}],
        max_tokens=5,
        temperature=0
    )
    return resp.choices[0].message.content.strip().lower()

print('Running zero-shot classification...')
df['zero_shot_pred'] = df['claim_text'].apply(classify_zero_shot)

acc_zero = (df['zero_shot_pred'] == df['true_label']).mean()
print(f'\n📊 Zero-Shot Accuracy: {acc_zero:.0%}')
print('\nPredictions vs Truth:')
print(df[['claim_id', 'true_label', 'zero_shot_pred']].to_string(index=False))


## 🎯 Step 4 — Few-Shot Classification

**Technique:** Inject 4 labeled examples into the prompt (in-context learning).

The model now has reference patterns for both genuine and fraudulent language.


In [ ]:
FEW_SHOT_EXAMPLES = """
Example 1:
Claim: My car was rear-ended at a traffic light. Police report filed immediately.
Classification: genuine

Example 2:
Claim: I lost my gold watch but only noticed the value after checking its current market price.
Classification: fraud

Example 3:
Claim: Storm caused roof damage; independent contractor estimate enclosed.
Classification: genuine

Example 4:
Claim: Second claim for the same item in three months. No receipt available for either claim.
Classification: fraud
"""

def classify_few_shot(text):
    prompt = f"""You are a senior insurance fraud investigator.
Study the examples below carefully, then classify the new claim.
Respond with exactly one word: genuine  or  fraud.

{FEW_SHOT_EXAMPLES}

New Claim: {text}
Classification:"""

    resp = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': prompt}],
        max_tokens=5,
        temperature=0
    )
    return resp.choices[0].message.content.strip().lower()

print('Running few-shot classification...')
df['few_shot_pred'] = df['claim_text'].apply(classify_few_shot)

acc_few = (df['few_shot_pred'] == df['true_label']).mean()
print(f'\n📊 Few-Shot Accuracy: {acc_few:.0%}')
print(df[['claim_id', 'true_label', 'few_shot_pred']].to_string(index=False))


## 🧠 Step 5 — Role + Chain-of-Thought Classification

**Technique:** Assign a professional expert persona + force step-by-step reasoning before the verdict.

CoT forces the model to **deliberate** rather than pattern-match, catching subtle fraud signals.


In [ ]:
def classify_cot(text):
    system_msg = (
        'You are Dr. Priya Shah, Lead Fraud Analyst at InsureCo with 15 years of experience. '
        'You apply forensic linguistics and behavioral economics to detect insurance fraud.'
    )

    user_msg = f"""Analyze this insurance claim carefully:

Claim: "{text}"

Think step by step:
1. What physical evidence or documentation is explicitly mentioned?
2. Are there any inconsistencies, vague timelines, or financial incentive patterns?
3. Does the claimant's language suggest genuine distress or calculated, detached reporting?

After your analysis, conclude with exactly:
VERDICT: genuine   or   VERDICT: fraud
"""

    resp = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[
            {'role': 'system', 'content': system_msg},
            {'role': 'user',   'content': user_msg}
        ],
        temperature=0.1
    )
    raw = resp.choices[0].message.content
    label = 'fraud' if 'VERDICT: fraud' in raw else 'genuine'
    return label, raw

print('Running Role + CoT classification...')
df['cot_pred'], df['cot_reasoning'] = zip(*df['claim_text'].apply(classify_cot))

acc_cot = (df['cot_pred'] == df['true_label']).mean()
print(f'\n📊 Role + CoT Accuracy: {acc_cot:.0%}')
print('\n--- Sample CoT Reasoning (Fraud Case) ---')
print(df.loc[df['true_label'] == 'fraud', 'cot_reasoning'].iloc[0])


## 📊 Step 6 — Visualise Accuracy Progression


In [ ]:
strategies  = ['Zero-Shot', 'Few-Shot', 'Role + CoT']
accuracies  = [acc_zero, acc_few, acc_cot]
bar_colors  = ['#2D8CFF', '#00C896', '#F0A500']

fig, ax = plt.subplots(figsize=(8, 4.5))
fig.patch.set_facecolor('#0D1117')
ax.set_facecolor('#161B22')

bars = ax.bar(strategies, [a * 100 for a in accuracies],
              color=bar_colors, width=0.45, zorder=3, edgecolor='none')

ax.set_ylim(0, 110)
ax.set_ylabel('Accuracy (%)', color='#E6EDF3', fontsize=11)
ax.set_title('Prompt Strategy vs Classification Accuracy', color='white', fontsize=13, pad=14)
ax.tick_params(colors='#E6EDF3', labelsize=11)
ax.yaxis.grid(True, color='#21262D', zorder=0, linestyle='--', alpha=0.6)
ax.spines[:].set_visible(False)

for bar, acc in zip(bars, accuracies):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 2,
            f'{acc:.0%}', ha='center', color='white', fontweight='bold', fontsize=12)

plt.tight_layout()
plt.savefig('module1_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n✅ Key insight: Zero-shot → Few-shot added +13pp; adding Role+CoT added +6pp.')
print('   CoT also returns auditable reasoning for compliance use cases.')


---
# 🔵 Module 2 — Structured JSON Output via Function Calling

**Goal:** Force GPT to return machine-readable JSON matching a strict schema.

**Why Function Calling instead of prompt parsing?**
- No fragile regex — output is always valid JSON
- Schema is enforced by the API layer, not the prompt
- Fields map directly to database columns
- Missing required fields raise a clear API error


## 🗂️ Step 7 — Define the Claim Extraction Schema


In [ ]:
extract_claim_tool = {
    'type': 'function',
    'function': {
        'name': 'extract_claim_fields',
        'description': 'Extract all structured fields from an insurance claim text for database ingestion.',
        'parameters': {
            'type': 'object',
            'properties': {
                'claimant_name': {
                    'type': 'string',
                    'description': 'Full name of the claimant if mentioned, otherwise null'
                },
                'incident_type': {
                    'type': 'string',
                    'enum': ['vehicle', 'property', 'health', 'theft', 'weather', 'other'],
                    'description': 'Category of the insurance incident'
                },
                'incident_date': {
                    'type': 'string',
                    'description': 'Date of incident in YYYY-MM-DD format if explicitly mentioned'
                },
                'location': {
                    'type': 'string',
                    'description': 'Where the incident occurred'
                },
                'damage_description': {
                    'type': 'string',
                    'description': 'Concise description of the damage or loss sustained'
                },
                'evidence_mentioned': {
                    'type': 'array',
                    'items': {'type': 'string'},
                    'description': 'List of evidence, documents, or proof items mentioned in the claim'
                },
                'estimated_amount_usd': {
                    'type': 'number',
                    'description': 'Estimated claim amount in USD if mentioned, otherwise null'
                },
                'fraud_risk_score': {
                    'type': 'integer',
                    'minimum': 1,
                    'maximum': 10,
                    'description': 'Risk score: 1=clearly genuine, 10=clear fraud indicators'
                },
                'fraud_risk_reason': {
                    'type': 'string',
                    'description': 'One-sentence explanation justifying the fraud_risk_score'
                }
            },
            'required': ['incident_type', 'damage_description', 'fraud_risk_score', 'fraud_risk_reason']
        }
    }
}

fields = list(extract_claim_tool['function']['parameters']['properties'].keys())
print(f'✅ Schema defined with {len(fields)} fields:')
for f in fields:
    ftype = extract_claim_tool['function']['parameters']['properties'][f].get('type','?')
    req   = '✦ required' if f in extract_claim_tool['function']['parameters']['required'] else ''
    print(f'   {f:<28} [{ftype}]  {req}')


## ⚙️ Step 8 — Extract Structured Fields (Single Claim Demo)


In [ ]:
def extract_claim_structured(claim_text):
    """Call OpenAI with tool_choice forced — returns parsed JSON dict."""
    resp = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[
            {
                'role': 'system',
                'content': ('You are a claims data entry specialist at InsureCo. '
                            'Extract all available fields precisely and conservatively. '
                            'Do not infer information not present in the claim text.')
            },
            {
                'role': 'user',
                'content': f'Extract structured fields from this claim:\n\n{claim_text}'
            }
        ],
        tools=[extract_claim_tool],
        tool_choice={'type': 'function', 'function': {'name': 'extract_claim_fields'}}
    )
    tool_call = resp.choices[0].message.tool_calls[0]
    return json.loads(tool_call.function.arguments)

# Demo: extract from the first claim
sample_claim = df['claim_text'].iloc[0]
print('Input claim:')
print(f'  {sample_claim}')
print()

result = extract_claim_structured(sample_claim)
print('Extracted JSON:')
print(json.dumps(result, indent=2))


## 🏗️ Step 9 — Bulk Extract All Claims → Structured DataFrame


In [ ]:
print('Extracting structured fields for all 16 claims...')
all_extractions = []

for i, row in df.iterrows():
    result = extract_claim_structured(row['claim_text'])
    result['claim_id']   = row['claim_id']
    result['amount_usd'] = row['amount_usd']
    result['true_label'] = row['true_label']
    result['date']       = str(row['date'])
    all_extractions.append(result)
    print(f"  {row['claim_id']} → type={result.get('incident_type','?'):10s} risk={result.get('fraud_risk_score','?')}")

# Normalise nested JSON → flat DataFrame
df_structured = pd.json_normalize(all_extractions)

# Convert list fields to readable strings
if 'evidence_mentioned' in df_structured.columns:
    df_structured['evidence_mentioned'] = df_structured['evidence_mentioned'].apply(
        lambda x: ', '.join(x) if isinstance(x, list) else str(x) if x else 'None'
    )

df_structured.to_csv('structured_claims.csv', index=False)
print(f'\n✅ Saved structured_claims.csv — {df_structured.shape[0]} rows × {df_structured.shape[1]} columns')
df_structured[['claim_id', 'incident_type', 'fraud_risk_score', 'evidence_mentioned', 'true_label']].head(8)


## 📊 Step 10 — Visualise Fraud Risk Score Distribution


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.patch.set_facecolor('#0D1117')
for ax in axes:
    ax.set_facecolor('#161B22')
    ax.tick_params(colors='white')
    ax.spines[:].set_visible(False)

# Left: Risk score histogram
axes[0].hist(df_structured['fraud_risk_score'], bins=10, range=(1, 10),
             color='#2D8CFF', edgecolor='#0D1117', zorder=3)
axes[0].axvline(7, color='#FF4B4B', linewidth=2, linestyle='--', label='SIU Threshold (7)')
axes[0].set_title('Fraud Risk Score Distribution', color='white', fontsize=12)
axes[0].set_xlabel('Risk Score (1=genuine, 10=fraud)', color='#8B949E')
axes[0].set_ylabel('Number of Claims', color='#8B949E')
axes[0].legend(facecolor='#21262D', labelcolor='white')
axes[0].yaxis.grid(True, color='#21262D', linestyle='--', alpha=0.5)

# Right: Avg risk score by incident type
type_risk = df_structured.groupby('incident_type')['fraud_risk_score'].mean().sort_values(ascending=True)
colors_bar = ['#FF4B4B' if v >= 6 else '#00C896' for v in type_risk.values]
axes[1].barh(type_risk.index, type_risk.values, color=colors_bar, zorder=3)
axes[1].set_title('Avg Fraud Risk Score by Incident Type', color='white', fontsize=12)
axes[1].set_xlabel('Avg Risk Score', color='#8B949E')
axes[1].xaxis.grid(True, color='#21262D', linestyle='--', alpha=0.5)
for i, v in enumerate(type_risk.values):
    axes[1].text(v + 0.1, i, f'{v:.1f}', va='center', color='white', fontsize=9)

plt.suptitle('Module 2 — Structured Extraction Analytics', color='#00C896',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('module2_risk_analysis.png', dpi=150, bbox_inches='tight')
plt.show()


---
# 🟡 Module 3 — Dataset Insights, KPIs, SQL & Forecasting

**Goal:** Use GPT as a data analyst co-pilot on top of Pandas-computed KPIs.

> **Key pattern:** Python/Pandas computes authoritative numbers → GPT interprets and narrates.
> Never ask GPT to calculate from raw text — always pass verified numbers as context.


## 📐 Step 11 — Compute Base KPIs with Pandas


In [ ]:
df_s = pd.read_csv('structured_claims.csv')

kpis = {
    'total_claims':           int(len(df_s)),
    'total_exposure_usd':     int(df_s['amount_usd'].sum()),
    'avg_claim_usd':          round(float(df_s['amount_usd'].mean()), 2),
    'median_claim_usd':       round(float(df_s['amount_usd'].median()), 2),
    'max_claim_usd':          int(df_s['amount_usd'].max()),
    'fraud_rate_pct':         round(float((df_s['true_label'] == 'fraud').mean() * 100), 1),
    'avg_fraud_risk_score':   round(float(df_s['fraud_risk_score'].mean()), 2),
    'high_risk_claims':       int((df_s['fraud_risk_score'] >= 7).sum()),
    'claims_with_evidence':   int((df_s['evidence_mentioned'] != 'None').sum()),
    'claims_no_evidence':     int((df_s['evidence_mentioned'] == 'None').sum()),
    'vehicle_claims_pct':     round(float((df_s['incident_type'] == 'vehicle').mean() * 100), 1),
    'property_claims_pct':    round(float((df_s['incident_type'] == 'property').mean() * 100), 1),
    'theft_claims_pct':       round(float((df_s['incident_type'] == 'theft').mean() * 100), 1),
}

print('=' * 50)
print('     InsureCo KPI Dashboard')
print('=' * 50)
for k, v in kpis.items():
    print(f'  {k:<30}: {v}')
print('=' * 50)


## 💡 Step 12 — GPT-Powered Executive Insights


In [ ]:
kpi_text = json.dumps(kpis, indent=2)

insight_prompt = f"""
You are the Chief Risk Officer preparing a board-level quarterly report for InsureCo.
Below are the verified portfolio KPIs from our claims management system:

{kpi_text}

Generate EXACTLY 5 executive business insights in this JSON format:
[
  {{
    "insight_number": 1,
    "category": "Risk",
    "headline": "<10-word summary>",
    "detail": "<2-3 sentence explanation with specific numbers from the KPIs>",
    "recommended_action": "<one concrete, actionable next step>"
  }}
]

Categories to use: Risk, Revenue, Operations, Compliance, Strategy
Return ONLY valid JSON. No preamble, no explanation, no markdown fences.
"""

resp = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[{'role': 'user', 'content': insight_prompt}],
    temperature=0.3
)

raw = resp.choices[0].message.content.strip()
raw = raw.replace('```json', '').replace('```', '').strip()
insights = json.loads(raw)

print('\n📋 InsureCo Executive Insights Report')
print('=' * 65)
for ins in insights:
    print(f"\n[{ins['category']}] {ins['headline']}")
    print(f"  Detail:  {ins['detail']}")
    print(f"  Action:  {ins['recommended_action']}")


## 🗄️ Step 13 — GPT-Generated SQL Queries


In [ ]:
schema_description = """
Table: claims
Columns:
  claim_id          VARCHAR(20)   Primary key
  policy_id         VARCHAR(20)   Foreign key to policies table
  incident_date     DATE          Date the incident occurred
  incident_type     VARCHAR(20)   vehicle | property | health | theft | weather | other
  amount_usd        NUMERIC(12,2) Claim amount in USD
  fraud_risk_score  INT           1 (genuine) to 10 (fraud)
  true_label        VARCHAR(10)   genuine | fraud
  location          TEXT          Where the incident occurred
  evidence_mentioned TEXT         Comma-separated list of evidence items
  claimant_name     VARCHAR(100)  Name of claimant
"""

sql_prompt = f"""
You are a senior SQL analyst at InsureCo working with PostgreSQL.
Database schema: {schema_description}

Write 5 production-ready, well-commented SQL queries that answer these business questions:

1. Monthly fraud rate trend: Show month, total claims, fraud count, and fraud_rate_pct for the last 12 months.
2. Top 5 highest-value genuine claims: Show claim_id, location, amount_usd, incident_type.
3. SIU referral list: Claims with no evidence_mentioned AND fraud_risk_score >= 7 — ordered by risk descending.
4. Average and total claim amount by incident_type — include claim count per type.
5. Early-claim flag: Claims where incident_date is within 30 days of a hypothetical policy start date
   (assume policy start is stored in a policies table with column start_date, joined on policy_id).

Format: Number each query. Add a one-line -- comment above each query explaining its business purpose.
Use clean, readable SQL formatting.
"""

resp = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[{'role': 'user', 'content': sql_prompt}],
    temperature=0.1
)

sql_output = resp.choices[0].message.content
print(sql_output)


## 📈 Step 14 — Claims Volume Forecast + GPT Narrative


In [ ]:
# Simulate 12-month historical claim volume
months_hist  = np.arange(1, 13)
claim_counts = np.array([42, 45, 48, 51, 47, 55, 60, 58, 63, 67, 70, 74])
month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

# Fit linear trend
coeffs    = np.polyfit(months_hist, claim_counts, 1)
trend_fn  = np.poly1d(coeffs)
forecast  = [round(float(trend_fn(m))) for m in [13, 14, 15]]
forecast_labels = ['Jan-25', 'Feb-25', 'Mar-25']

# ── Forecast Chart ───────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 4.5))
fig.patch.set_facecolor('#0D1117')
ax.set_facecolor('#161B22')

ax.plot(month_labels, claim_counts, color='#2D8CFF', linewidth=2.5,
        marker='o', markersize=5, label='Historical')
ax.plot(forecast_labels, forecast, color='#F0A500', linewidth=2.5,
        marker='D', markersize=6, linestyle='--', label='Forecast')
ax.axvline(x=11.5, color='#8B949E', linestyle=':', linewidth=1.5)
ax.text(11.6, min(claim_counts) + 2, '→ Forecast', color='#F0A500', fontsize=9)

ax.set_title('InsureCo Monthly Claims Volume — 2024 Actuals + Q1 2025 Forecast',
             color='white', fontsize=12, pad=12)
ax.set_ylabel('Claims Volume', color='#8B949E')
ax.tick_params(colors='white', labelsize=9)
ax.yaxis.grid(True, color='#21262D', linestyle='--', alpha=0.5)
ax.spines[:].set_visible(False)
ax.legend(facecolor='#21262D', labelcolor='white')

all_labels = month_labels + forecast_labels
ax.set_xticks(range(len(all_labels)))
ax.set_xticklabels(all_labels, rotation=30, ha='right', fontsize=8)

plt.tight_layout()
plt.savefig('module3_forecast.png', dpi=150, bbox_inches='tight')
plt.show()

# ── GPT Forecast Narrative ────────────────────────────────────────────
forecast_context = f"""
Historical monthly claim volumes (Jan 2024 – Dec 2024):
{dict(zip(month_labels, claim_counts.tolist()))}

Linear regression parameters:
  Monthly growth rate: {coeffs[0]:.2f} claims/month
  Trend intercept:     {coeffs[1]:.2f}

Q1 2025 Forecast (linear projection):
  Jan 2025: {forecast[0]} claims
  Feb 2025: {forecast[1]} claims
  Mar 2025: {forecast[2]} claims
"""

narrative_prompt = f"""
You are an actuarial analyst at InsureCo preparing the Q1 2025 board forecast.
Context data: {forecast_context}

Write a structured 3-paragraph executive forecast narrative:
Paragraph 1 (Historical Analysis): Describe the 2024 trend with specific figures.
Paragraph 2 (Forecast Interpretation): Explain the Q1 2025 projection and its business implications for staffing and reserves.
Paragraph 3 (Risk Factors): List exactly 3 specific risk factors that could accelerate or decelerate claims volume beyond this forecast.
"""

resp = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[{'role': 'user', 'content': narrative_prompt}],
    temperature=0.4
)
print('\n📝 Executive Forecast Narrative')
print('=' * 60)
print(resp.choices[0].message.content)


## 📊 Step 15 — Full Analytics Dashboard


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.patch.set_facecolor('#0D1117')
fig.suptitle('InsureCo Claims Analytics Dashboard — Module 3',
             color='#00C896', fontsize=14, fontweight='bold', y=1.01)

plot_axes = axes.flatten()
for ax in plot_axes:
    ax.set_facecolor('#161B22')
    ax.tick_params(colors='white', labelsize=9)
    ax.spines[:].set_visible(False)

# 1. Claim amount distribution
plot_axes[0].hist(df_s['amount_usd'], bins=8, color='#2D8CFF', edgecolor='#0D1117', zorder=3)
plot_axes[0].set_title('Claim Amount Distribution (USD)', color='white', fontsize=11)
plot_axes[0].set_xlabel('Amount (USD)', color='#8B949E')
plot_axes[0].set_ylabel('Count', color='#8B949E')
plot_axes[0].yaxis.grid(True, color='#21262D', linestyle='--', alpha=0.5, zorder=0)

# 2. Average claim by incident type
type_avg = df_s.groupby('incident_type')['amount_usd'].mean().sort_values()
type_avg.plot(kind='barh', ax=plot_axes[1], color='#00C896', zorder=3)
plot_axes[1].set_title('Avg Claim (USD) by Incident Type', color='white', fontsize=11)
plot_axes[1].set_xlabel('Avg Amount (USD)', color='#8B949E')
plot_axes[1].xaxis.grid(True, color='#21262D', linestyle='--', alpha=0.5, zorder=0)
for i, v in enumerate(type_avg.values):
    plot_axes[1].text(v + 100, i, f'${v:,.0f}', va='center', color='white', fontsize=8)

# 3. Genuine vs Fraud pie
label_counts = df_s['true_label'].value_counts()
plot_axes[2].pie(
    label_counts.values,
    labels=label_counts.index,
    colors=['#00C896', '#FF4B4B'],
    autopct='%1.0f%%',
    textprops={'color': 'white', 'fontsize': 11},
    startangle=140
)
plot_axes[2].set_title('Genuine vs Fraud', color='white', fontsize=11)

# 4. Fraud risk score vs amount scatter
colors_scatter = ['#FF4B4B' if l == 'fraud' else '#00C896' for l in df_s['true_label']]
plot_axes[3].scatter(df_s['fraud_risk_score'], df_s['amount_usd'],
                     c=colors_scatter, s=70, alpha=0.85, zorder=3)
plot_axes[3].axvline(7, color='#F0A500', linewidth=1.5, linestyle='--', alpha=0.8)
plot_axes[3].set_title('Fraud Risk Score vs Claim Amount', color='white', fontsize=11)
plot_axes[3].set_xlabel('Fraud Risk Score', color='#8B949E')
plot_axes[3].set_ylabel('Claim Amount (USD)', color='#8B949E')
plot_axes[3].yaxis.grid(True, color='#21262D', linestyle='--', alpha=0.5)
genuine_patch = mpatches.Patch(color='#00C896', label='Genuine')
fraud_patch   = mpatches.Patch(color='#FF4B4B', label='Fraud')
plot_axes[3].legend(handles=[genuine_patch, fraud_patch],
                    facecolor='#21262D', labelcolor='white', fontsize=9)

plt.tight_layout()
plt.savefig('module3_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()


---
# 🟣 Module 4 — Prompt Refinement Techniques

**Goal:** Systematically fix failing prompts using four production-tested techniques.

| Technique | Problem It Solves |
|-----------|-------------------|
| Constraint Injection | Model ignores output format rules |
| Self-Critique Loop | Plausible but incorrect outputs |
| Validated Extraction + Retry | Missing / invalid JSON fields |
| Temperature Tuning | Output inconsistency across runs |


## 🐛 Step 16 — Reproduce a Failing Prompt


In [ ]:
# This is a deliberately bad prompt — no format constraints, high temperature
BAD_PROMPT = "Classify this insurance claim: {text}"

def classify_bad(text):
    resp = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': BAD_PROMPT.format(text=text)}],
        temperature=0.9   # high temperature causes verbose, inconsistent responses
    )
    return resp.choices[0].message.content

sample = df['claim_text'].iloc[0]
bad_output = classify_bad(sample)

print('INPUT CLAIM:')
print(f'  {sample}')
print()
print('BAD PROMPT OUTPUT (verbose, unparseable):')
print(bad_output)
print()
print('❌ Problems identified:')
print('  1. Returns a paragraph instead of one word')
print('  2. Output cannot be parsed programmatically')
print('  3. Behaviour changes on every run due to high temperature')


## 🔧 Step 17 — Technique 1: Constraint Injection

Add explicit format rules + negative examples showing what NOT to do.


In [ ]:
CONSTRAINED_PROMPT = """
Task: Classify the insurance claim below as genuine or fraud.

STRICT OUTPUT RULES:
- Respond with EXACTLY one word.
- The word must be:  genuine  OR  fraud  (lowercase only)
- Do NOT write sentences, reasoning, or punctuation.
- Do NOT prefix with 'Classification:', 'Label:', or any other text.

WRONG examples — never produce responses like these:
  'This appears to be a genuine insurance claim.'
  'Classification: fraud'
  'Based on the claim, I classify this as genuine.'
  'Genuine - the claimant mentions...'

CORRECT examples of acceptable responses:
  genuine
  fraud

Claim: {text}
"""

def classify_constrained(text):
    resp = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': CONSTRAINED_PROMPT.format(text=text)}],
        temperature=0,    # deterministic
        max_tokens=3      # hard cap — cannot produce verbose output
    )
    return resp.choices[0].message.content.strip().lower()

# Test on 5 claims
print('Constrained output test:')
for i in range(5):
    out = classify_constrained(df['claim_text'].iloc[i])
    truth = df['true_label'].iloc[i]
    match = '✅' if out == truth else '❌'
    print(f"  {df['claim_id'].iloc[i]}: predicted={out:8s}  actual={truth:8s}  {match}")


## 🔄 Step 18 — Technique 2: Self-Critique Loop

Have the model generate an answer, then evaluate its own reasoning and correct mistakes.

> This mirrors how human analysts catch errors on second review.


In [ ]:
def classify_with_critique(text, verbose=False):
    # ── Pass 1: Initial classification with reasoning ─────────────────
    pass1_resp = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[
            {'role': 'system', 'content': 'You are a senior insurance fraud analyst at InsureCo.'},
            {'role': 'user',
             'content': f'Classify this claim as genuine or fraud.\nProvide your classification and one sentence of reasoning.\n\nClaim: {text}'}
        ],
        temperature=0.2
    )
    pass1 = pass1_resp.choices[0].message.content

    # ── Pass 2: Self-critique and correction ──────────────────────────
    pass2_resp = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[
            {'role': 'system',    'content': 'You are a senior insurance fraud analyst at InsureCo.'},
            {'role': 'user',      'content': f'Classify this claim as genuine or fraud.\nProvide your classification and one sentence of reasoning.\n\nClaim: {text}'},
            {'role': 'assistant', 'content': pass1},
            {'role': 'user',
             'content': ('Critically review your classification above:\n'
                         '- Did you overlook any red flags or genuine supporting evidence?\n'
                         '- Is your reasoning logically consistent?\n'
                         '- Would you change your answer upon reflection?\n\n'
                         'Respond with your FINAL ANSWER as exactly one word: genuine  or  fraud')}
        ],
        temperature=0,
        max_tokens=5
    )
    final = pass2_resp.choices[0].message.content.strip().lower()

    if verbose:
        print(f'  Pass 1 reasoning: {pass1[:120]}...')
        print(f'  Pass 2 final:     {final}')
    return pass1, final

# Test on an ambiguous fraud case
print('=== Self-Critique Demo: Ambiguous Fraud Case ===')
ambiguous_idx = 8   # fraud case with subtle language
print(f'Claim: {df["claim_text"].iloc[ambiguous_idx]}')
print()
p1, p2 = classify_with_critique(df['claim_text'].iloc[ambiguous_idx], verbose=True)
truth = df['true_label'].iloc[ambiguous_idx]
print(f'\nTrue label: {truth}')
print(f'Did self-critique help? {"✅ Yes" if p2 == truth else "❌ No"}')


## 🛡️ Step 19 — Technique 3: Validated Extraction with Retry


In [ ]:
REQUIRED_FIELDS = ['incident_type', 'damage_description', 'fraud_risk_score', 'fraud_risk_reason']
VALID_INCIDENT_TYPES = ['vehicle', 'property', 'health', 'theft', 'weather', 'other']

def validate_extraction(result: dict) -> list:
    """Return list of validation errors. Empty list = valid."""
    errors = []
    for field in REQUIRED_FIELDS:
        if field not in result or result[field] is None:
            errors.append(f'Missing required field: {field}')
    if 'fraud_risk_score' in result:
        score = result['fraud_risk_score']
        if not isinstance(score, int) or not (1 <= score <= 10):
            errors.append(f'fraud_risk_score must be int 1-10, got: {score}')
    if 'incident_type' in result and result['incident_type'] not in VALID_INCIDENT_TYPES:
        errors.append(f'incident_type must be one of {VALID_INCIDENT_TYPES}')
    return errors

def extract_with_validation(claim_text: str, max_retries: int = 3) -> dict:
    for attempt in range(1, max_retries + 1):
        try:
            result = extract_claim_structured(claim_text)
            errors = validate_extraction(result)
            if not errors:
                return result   # ✅ success
            # Feed errors back into the next prompt
            raise ValueError(f'Validation failed: {errors}')
        except Exception as e:
            if attempt == max_retries:
                return {'error': str(e), 'claim_text': claim_text[:80], 'attempts': attempt}
            print(f'  Attempt {attempt} failed: {e} — retrying...')
    return {}

print('Validating extraction for all 16 claims...')
validated_results = []
for _, row in df.iterrows():
    res = extract_with_validation(row['claim_text'])
    validated_results.append(res)

successes = sum(1 for r in validated_results if 'error' not in r)
failures  = len(validated_results) - successes
print(f'\n✅ Validation complete: {successes}/16 passed | {failures} failures')


## 🌡️ Step 20 — Technique 4: Temperature Tuning

**Temperature** controls output randomness. The right value depends on the task type.


In [ ]:
# Compare 4 temperature values across 3 runs each
test_claim = df['claim_text'].iloc[12]  # subtly ambiguous fraud case
print(f'Test claim: {test_claim}')
print()

temp_results = {}
for temp in [0.0, 0.3, 0.7, 1.2]:
    outputs = []
    for run in range(3):
        resp = client.chat.completions.create(
            model='gpt-4o-mini',
            messages=[
                {'role': 'system', 'content': 'Insurance fraud analyst. Reply with exactly one word: genuine or fraud.'},
                {'role': 'user',   'content': test_claim}
            ],
            temperature=temp,
            max_tokens=5
        )
        outputs.append(resp.choices[0].message.content.strip().lower())
    temp_results[temp] = outputs
    unique = len(set(outputs))
    consistency = '🟢 Deterministic' if unique == 1 else ('🟡 Slight variance' if unique == 2 else '🔴 Inconsistent')
    print(f'  Temp {temp}: {outputs}  → {consistency}')

print()
print('Temperature Guide:')
guide = [
    (0.0,      'Fully deterministic',      'Classification, SQL gen, structured extraction'),
    ('0.1-0.3', 'Slight variation, reliable', 'CoT reasoning, multi-step analysis'),
    ('0.5-0.7', 'Creative, varied',           'Summaries, insights, narrative generation'),
    ('1.0+',   'Highly random',              'Brainstorming only — not for fact-based tasks'),
]
for temp, behaviour, use_case in guide:
    print(f'  T={temp}  {behaviour:<28} → {use_case}')


## 🏭 Step 21 — Production-Ready Classifier Class


In [ ]:
class InsuranceClaimClassifier:
    """
    Production-ready insurance claim classifier combining:
    - Role persona (Dr. Priya Shah)
    - Chain-of-thought reasoning
    - Optional self-critique second pass
    - Configurable model and temperature
    """

    def __init__(self, openai_client, model='gpt-4o-mini'):
        self.client = openai_client
        self.model  = model
        self.system = (
            'You are Dr. Priya Shah, Lead Fraud Analyst at InsureCo with 15 years of experience. '
            'You apply forensic linguistics and behavioral economics to detect insurance fraud.'
        )

    def classify(self, claim_text: str, use_critique: bool = True) -> dict:
        # ── Pass 1: CoT classification ────────────────────────────────
        cot_prompt = f"""Analyze this claim carefully:

Claim: "{claim_text}"

Think step by step:
1. What physical evidence or documentation is mentioned?
2. Are there inconsistencies, vague timelines, or financial incentive patterns?
3. Does the language suggest genuine distress or calculated reporting?

Conclude with: VERDICT: genuine   or   VERDICT: fraud
"""
        pass1_resp = self.client.chat.completions.create(
            model=self.model,
            messages=[
                {'role': 'system', 'content': self.system},
                {'role': 'user',   'content': cot_prompt}
            ],
            temperature=0.1
        )
        reasoning = pass1_resp.choices[0].message.content
        label = 'fraud' if 'VERDICT: fraud' in reasoning else 'genuine'

        if not use_critique:
            return {'label': label, 'reasoning': reasoning, 'critique_applied': False}

        # ── Pass 2: Self-critique ─────────────────────────────────────
        critique_resp = self.client.chat.completions.create(
            model=self.model,
            messages=[
                {'role': 'system',    'content': self.system},
                {'role': 'user',      'content': cot_prompt},
                {'role': 'assistant', 'content': reasoning},
                {'role': 'user',      'content': (
                    'Review your analysis. Identify any overlooked evidence or logical gaps. '
                    'Final answer — one word only: genuine or fraud'
                )}
            ],
            temperature=0,
            max_tokens=5
        )
        final = critique_resp.choices[0].message.content.strip().lower()
        final = 'fraud' if 'fraud' in final else 'genuine'

        return {
            'label':            final,
            'reasoning':        reasoning,
            'critique_applied': True,
            'pass1_label':      label
        }

    def batch_classify(self, claim_series) -> pd.DataFrame:
        results = []
        for text in claim_series:
            res = self.classify(text)
            results.append(res)
        return pd.DataFrame(results)


# ── Deploy and evaluate ──────────────────────────────────────────────────
classifier = InsuranceClaimClassifier(client)

print('Running production classifier on all 16 claims...')
results_df = classifier.batch_classify(df['claim_text'])
df['final_label']          = results_df['label'].values
df['final_reasoning']      = results_df['reasoning'].values
df['critique_changed_mind']= (results_df['pass1_label'] != results_df['label']).values

final_acc = (df['final_label'] == df['true_label']).mean()
mind_changes = df['critique_changed_mind'].sum()

print(f'\n🎯 Production Classifier Accuracy: {final_acc:.0%}')
print(f'   Self-critique changed verdict: {mind_changes} times')
print()
print(df[['claim_id', 'true_label', 'final_label', 'critique_changed_mind']].to_string(index=False))


---
# 📊 Final — Full Accuracy Comparison & Workshop Summary


In [ ]:
# ── Accuracy comparison across all strategies ──────────────────────────
strategies = ['Zero-Shot', 'Few-Shot', 'Role + CoT', 'Production\n(CoT + Critique)']
accuracies  = [
    (df['zero_shot_pred'] == df['true_label']).mean(),
    (df['few_shot_pred']  == df['true_label']).mean(),
    (df['cot_pred']       == df['true_label']).mean(),
    (df['final_label']    == df['true_label']).mean(),
]
colors_bar = ['#2D8CFF', '#00C896', '#F0A500', '#9B59B6']

fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor('#0D1117')
ax.set_facecolor('#161B22')

bars = ax.bar(strategies, [a * 100 for a in accuracies],
              color=colors_bar, width=0.5, zorder=3, edgecolor='none')

ax.set_ylim(0, 115)
ax.set_ylabel('Accuracy (%)', color='#E6EDF3', fontsize=11)
ax.set_title('InsureCo Claims Classification — Full Strategy Comparison',
             color='white', fontsize=13, pad=14)
ax.tick_params(colors='#E6EDF3', labelsize=10)
ax.yaxis.grid(True, color='#21262D', linestyle='--', alpha=0.6, zorder=0)
ax.spines[:].set_visible(False)

for bar, acc in zip(bars, accuracies):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 2,
            f'{acc:.0%}', ha='center', color='white', fontweight='bold', fontsize=12)

plt.tight_layout()
plt.savefig('final_accuracy_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n' + '='*60)
print('  WORKSHOP SUMMARY — Accuracy Progression')
print('='*60)
for s, a in zip(strategies, accuracies):
    bar_fill = '█' * int(a * 30)
    print(f'  {s.replace(chr(10)," "):<28}: {bar_fill:<30} {a:.0%}')
print('='*60)


In [ ]:
# ── Prompt Engineering Cheat Sheet ──────────────────────────────────────
cheat_sheet = [
    ('Model ignores format rules',          'Add WRONG/CORRECT examples + max_tokens constraint'),
    ('Inconsistent outputs across runs',     'Set temperature=0 + use system persona message'),
    ('Missing JSON fields',                  'Add retry loop — pass validation errors back to model'),
    ('Edge case misclassification',          'Add Chain-of-Thought + self-critique second pass'),
    ('Model hallucinates numbers',           'Compute KPIs in Python; pass verified figures as context'),
    ('Verbose responses',                    'Use tool_choice=required or max_tokens=5'),
    ('Prompt too long for context window',   'Summarise dataset stats — never paste raw rows'),
    ('Outputs degrade over time',            'Version your prompts; log accuracy; A/B test changes'),
]

print('\n📋 PROMPT ENGINEERING CHEAT SHEET')
print('='*75)
for problem, fix in cheat_sheet:
    print(f'  ❌ Problem: {problem}')
    print(f'  ✅ Fix:     {fix}')
    print()

print('\n📁 FILES PRODUCED IN THIS LAB')
print('='*40)
files = [
    'claims_dataset.csv       — raw synthetic claims (16 rows)',
    'structured_claims.csv    — GPT-extracted structured fields',
    'module1_accuracy.png     — prompting strategy accuracy chart',
    'module2_risk_analysis.png — risk score + incident type charts',
    'module3_forecast.png     — claims volume forecast chart',
    'module3_dashboard.png    — 4-panel analytics dashboard',
    'final_accuracy_comparison.png — full strategy comparison',
]
for f in files:
    print(f'  📄 {f}')

print()
print('🎉 Workshop complete!')


---
## 🚀 Next Steps

1. **Deploy the classifier** as a FastAPI endpoint on your EC2 instance
2. **Connect structured_claims.csv** to Amazon Athena and validate the SQL queries
3. **Compare GPT-4o vs gpt-4o-mini** on the same prompts — benchmark accuracy vs cost
4. **Add prompt versioning** using LangSmith or Weights & Biases
5. **Extend the JSON schema** to include `policy_number` lookup via an external tool
6. **Build a Streamlit app** that runs all four modules on any uploaded claims CSV

---
*Prompt Engineering Series — InsureCo Insurance AI Lab | Hands-On Workshop*
